# Scenario S2: HVI-CIDNet <span style='font-size: 0.6em; vertical-align: middle; padding: 4px 8px; background-color: #FF5722; color: white; border-radius: 4px; margin-left: 10px;'>KAGGLE-NEW</span>
---
**Advanced Pipeline architecture** optimized for Kaggle environment with pre-enhanced dataset support.

--- 
 ## 0. Setup

In [ ]:
#@title 0.1 · Environment Setup & Clone Repo
import os, subprocess, sys, shutil

QUICK_TEST  = False  # @param {type:"boolean"}
REPO_URL    = "https://github.com/Otachiking/Object-Detection-ExDARK-with-LLIE.git"
SCENARIO_KEY  = "s2_hvi_cidnet"
SCENARIO_NAME = "S2_HVI_CIDNet"

RESTORE_PREVIOUS   = True   # @param {type:"boolean"}
GDRIVE_WEIGHTS_URL = ""     # @param {type:"string"}

_IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
_IS_COLAB  = not _IS_KAGGLE and os.path.exists("/content")

if _IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/TA-IQBAL-ObjectDetectionExDARKwithLLIE"
elif _IS_KAGGLE:
    REPO_DIR = "/kaggle/working/TA-IQBAL-ObjectDetectionExDARKwithLLIE"
else:
    raise RuntimeError("Run this on Kaggle or Colab")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)


In [ ]:
#@title 0.2 · Install Dependencies
!pip install -q ultralytics pyiqa thop fvcore scipy pandas pyyaml seaborn tqdm gdown huggingface_hub


In [ ]:
#@title 0.3 · Load Configuration & Define Paths
from src.config import load_config, save_environment_info
from src.seed import set_global_seed

cfg = load_config(SCENARIO_KEY, quick_test=QUICK_TEST, epochs=EPOCHS)
set_global_seed(cfg["seed"])

OUTPUT_ROOT = cfg.get("paths", {}).get("output_root") or cfg.get("paths", {}).get("project_root")
EXDARK_ROOT = cfg.get("paths", {}).get("exdark_root") or cfg.get("paths", {}).get("data", {}).get("exdark_original")

ENHANCED_IN = None
# Auto-detect for Kaggle if needed
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    from src.utils.platform import resolve_kaggle_exdark, resolve_kaggle_llie_input, get_kaggle_enhanced_input
    _classlist_check = os.path.join(EXDARK_ROOT or "", "Groundtruth", "imageclasslist.txt")
    if not EXDARK_ROOT or not os.path.isfile(_classlist_check):
        _detected = resolve_kaggle_exdark()
        if _detected:
            EXDARK_ROOT = _detected
    llie_in = resolve_kaggle_llie_input()
    if llie_in:
        cfg["paths"]["model_weights_input"] = llie_in
    
    # Search for Pre-Enhanced Dataset
    ENHANCED_IN = get_kaggle_enhanced_input(cfg.get("scenario", {}).get("enhancer", ""))

print(f"Output root : {OUTPUT_ROOT}")
print(f"ExDark root : {EXDARK_ROOT}")
if ENHANCED_IN: print(f"Enhanced IN: {ENHANCED_IN}")

PREPARED_DIR   = os.path.join(OUTPUT_ROOT, "prepared")
SCENARIO_DIR   = os.path.join(OUTPUT_ROOT, "scenarios", SCENARIO_NAME)
SCENARIO_RUNS  = os.path.join(SCENARIO_DIR, "runs")
SCENARIO_EVAL  = os.path.join(SCENARIO_DIR, "evaluation")

os.makedirs(PREPARED_DIR, exist_ok=True)
os.makedirs(SCENARIO_RUNS, exist_ok=True)
os.makedirs(SCENARIO_EVAL, exist_ok=True)
save_environment_info(SCENARIO_DIR)

--- 
 ## Fase 1: Data Preparation

In [ ]:
#@title Fase 1 · Data Preparation (auto-skip if done)
from src.data.split_dataset     import parse_split_file
from src.data.convert_exdark    import convert_exdark_to_yolo
from src.data.build_yolo_dataset import build_yolo_dataset

img_dir        = os.path.join(EXDARK_ROOT, "Dataset")
gt_dir         = os.path.join(EXDARK_ROOT, "Groundtruth")
split_output   = os.path.join(PREPARED_DIR, "splits")
labels_dir     = os.path.join(PREPARED_DIR, "ExDark_yolo_labels")
yolo_dir       = os.path.join(PREPARED_DIR, "ExDark_yolo")

splits = parse_split_file(os.path.join(gt_dir, "imageclasslist.txt"), split_output)
stats = convert_exdark_to_yolo(img_dir, gt_dir, labels_dir)
build_stats = build_yolo_dataset(img_dir, labels_dir, split_output, yolo_dir, target_size=cfg["yolo"]["imgsz"])


--- 
 ## Fase 2: Image Enhancement

In [ ]:
#@title Fase 2 · Image Enhancement
import torch
from src.enhancement.run_enhancement import enhance_dataset, get_enhancer
from src.utils.platform import setup_enhanced_from_kaggle

enhancer_name = cfg.get("scenario", {}).get("enhancer")
enhanced_dir = os.path.join(SCENARIO_DIR, "enhanced")
cache_dir = os.path.join(OUTPUT_ROOT, "model_cache")

skip_enhancement = False
if ENHANCED_IN:
    # yolo_dir is defined in Fase 1
    skip_enhancement = setup_enhanced_from_kaggle(ENHANCED_IN, enhanced_dir, yolo_dir)

if not skip_enhancement:
    enhancer = get_enhancer(enhancer_name, cache_dir)
    enhancer.load_model()
    stats = enhance_dataset(enhancer=enhancer, source_dataset_dir=yolo_dir, output_dir=enhanced_dir, yolo_labels_dir=yolo_dir)
else:
    print(f"[SKIP] Using pre-enhanced images from Kaggle: {ENHANCED_IN}")


--- 
 ## Fase 2.5: Sample Test Images — Original vs Enhanced

In [ ]:
#@title Fase 2.5 · Sample Test Images Preview
from src.evaluation.visualize import plot_sample_images
from IPython.display import Image, display

saved_path = plot_sample_images(
    raw_test_dir=os.path.join(PREPARED_DIR, "ExDark_yolo", "images", "test"),
    enh_test_dir=os.path.join(SCENARIO_DIR, "enhanced", "images", "test") if enhancer_name else None,
    output_dir=SCENARIO_EVAL,
    scenario_name=SCENARIO_NAME,
    enhancer_name=enhancer_name
)
if saved_path and os.path.exists(saved_path):
    display(Image(filename=saved_path))


--- 
 ## Fase 3: Image Quality Metrics

In [ ]:
#@title Fase 3 · Image Quality Metrics
# DEFAULT FORCE TO TRUE based on requirements
FORCE_EVALUATION = True # @param {type:"boolean"}
from src.evaluation.nr_metrics import compute_nr_metrics

raw_test_dir = os.path.join(PREPARED_DIR, "ExDark_yolo", "images", "test")
test_dir = os.path.join(SCENARIO_DIR, "enhanced", "images", "test") if enhancer_name else raw_test_dir

nr = compute_nr_metrics(
    images_dir=test_dir, 
    raw_images_dir=raw_test_dir, 
    output_dir=SCENARIO_EVAL, 
    scenario_name=SCENARIO_NAME, 
    force=FORCE_EVALUATION
)
print(f"\nNR-IQA (lower is better) — {SCENARIO_NAME}")
print(f"  NIQE    : {nr.get('niqe_mean')}")
print(f"  BRISQUE : {nr.get('brisque_mean')}")
print(f"  LOE     : {nr.get('loe_mean')}")

# --- NEW SPATIAL METRICS ---
print("\n" + "="*60)
from src.evaluation.spatial_metrics import compute_and_show_spatial_metrics
from IPython.display import Image, display

spatial_summ, spatial_img = compute_and_show_spatial_metrics(
    test_dir=test_dir,
    output_dir=SCENARIO_EVAL,
    scenario_name=SCENARIO_NAME
)
if spatial_img and os.path.exists(spatial_img):
    display(Image(filename=spatial_img))


--- 
 ## Fase 4: Training

In [ ]:
#@title Fase 4 · Training
# DEFAULT FORCE TO TRUE (best.pt ditimpa selalu kecuali skip manual)
FORCE_RETRAIN = True # @param {type:"boolean"}
USE_PRETRAINED = True # @param {type:"boolean"}
from src.training.train_yolo import train_yolo, get_best_weights

data_yaml = os.path.join(SCENARIO_DIR, "enhanced", "dataset.yaml") if enhancer_name else os.path.join(PREPARED_DIR, "ExDark_yolo", "dataset.yaml")

result = train_yolo(
    dataset_yaml=data_yaml, 
    scenario_name=SCENARIO_NAME, 
    output_dir=SCENARIO_DIR, 
    run_name="runs", 
    config=cfg, 
    force=FORCE_RETRAIN, 
    use_pretrained=USE_PRETRAINED
)
best_pt = get_best_weights(SCENARIO_RUNS)


--- 
 ## Fase 4.5: Training Curves & Figures

In [ ]:
#@title Fase 4.5 · Training Curves & Figures
from src.evaluation.visualize import plot_training_curves
from IPython.display import Image, display

curve_paths = plot_training_curves(run_dir_train=SCENARIO_RUNS, output_dir=SCENARIO_EVAL, scenario_name=SCENARIO_NAME)
for p in curve_paths:
    if p and os.path.exists(p):
        display(Image(filename=p))


--- 
 ## Fase 5: Detection Evaluation

In [ ]:
#@title Fase 5 · Detection Evaluation
# Evaluasi otomatis retrain jika FORCE_EVALUATION True
from src.evaluation.eval_yolo import evaluate_yolo
import glob
from IPython.display import Image, display

data_yaml = os.path.join(SCENARIO_DIR, "enhanced", "dataset.yaml") if enhancer_name else os.path.join(PREPARED_DIR, "ExDark_yolo", "dataset.yaml")
best_pt = get_best_weights(SCENARIO_RUNS)

results = evaluate_yolo(weights_path=best_pt, dataset_yaml=data_yaml, output_dir=SCENARIO_EVAL, scenario_name=SCENARIO_NAME, force=FORCE_EVALUATION)
print(results.get("overall", {}))

val_plots_dir = os.path.join(SCENARIO_EVAL, "val_plots")
from src.evaluation.visualize import plot_validation_batches, plot_confusion_matrices

if os.path.exists(val_plots_dir):
    print("\n[Visualisasi] Menampilkan Plot Validasi YOLO (Confusion Matrix & Batches):")
    
    # 1. Confusion Matrix Grid
    cm_path = plot_confusion_matrices(val_plots_dir, SCENARIO_EVAL, SCENARIO_NAME)
    if cm_path and os.path.exists(cm_path):
        display(Image(filename=cm_path))
        
    # 2. Validation Batches Grid
    batches_path = plot_validation_batches(val_plots_dir, SCENARIO_EVAL, SCENARIO_NAME)
    if batches_path and os.path.exists(batches_path):
        display(Image(filename=batches_path))
        
    # 3. All other F1/PR Curves
    for f in sorted(glob.glob(os.path.join(val_plots_dir, "*_curve.png"))):
        display(Image(filename=f))


--- 
 ## Fase 5.5: Detection Visualization (GT vs Prediction)

In [ ]:
#@title Fase 5.5 · Detection Visualization (GT vs Prediction)
from src.evaluation.visualize import plot_detection_results
from IPython.display import Image, display

test_img_dir = os.path.join(SCENARIO_DIR, "enhanced", "images", "test") if enhancer_name else os.path.join(PREPARED_DIR, "ExDark_yolo", "images", "test")
test_lbl_dir = os.path.join(PREPARED_DIR, "ExDark_yolo", "labels", "test")

det_path = plot_detection_results(
    weights_path=best_pt,
    test_img_dir=test_img_dir,
    test_lbl_dir=test_lbl_dir,
    output_dir=SCENARIO_EVAL,
    scenario_name=SCENARIO_NAME
)
if det_path and os.path.exists(det_path):
    display(Image(filename=det_path))


--- 
 ## Fase 5.6: Model Interpretability (What YOLO Sees)

In [ ]:
#@title Fase 5.6 · Model Interpretability (Edge & Feature Maps)
from src.evaluation.interpretability import generate_and_plot_yolo_vision
from IPython.display import Image, display

test_img_dir = os.path.join(SCENARIO_DIR, "enhanced", "images", "test") if enhancer_name else os.path.join(PREPARED_DIR, "ExDark_yolo", "images", "test")

print("\n🔍 MEMBANGUN VISUALISASI FEATURE MAPS (WHAT YOLO SEES)")
print("="*60)
# Menggunakan model terbaik yang sudah ditraining
viz_paths = generate_and_plot_yolo_vision(
    weights_path=best_pt,
    test_images_dir=test_img_dir,
    output_dir=SCENARIO_EVAL,
    scenario_name=SCENARIO_NAME
)

if viz_paths:
    for vp in viz_paths:
        display(Image(filename=vp))


--- 
 ## Fase 6: Latency & FLOPs

In [ ]:
#@title Fase 6 · Latency & FLOPs
from src.evaluation.latency import measure_latency
from src.evaluation.flops import compute_all_flops

raw_test_dir = os.path.join(PREPARED_DIR, "ExDark_yolo", "images", "test")

lat = measure_latency(yolo_weights=best_pt, output_dir=SCENARIO_EVAL, scenario_name=SCENARIO_NAME, test_images_dir=raw_test_dir)
flops = compute_all_flops(yolo_weights=best_pt, output_dir=SCENARIO_EVAL, scenario_name=SCENARIO_NAME)


In [ ]:
#@title Done
print("Notebook finished running all modular phases.")
